# Упражнение 3, част 1: линейна регресия

**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Цел на упражнението](#cel)
2. [Файлове в упражнението](#failove)
3. [Подготовка и стартиране](#podgotovka)
4. [Част 1. Дилемата на затворника](#chast-1)
5. [Част 2. Стратегии и турнир](#chast-2)
6. [Част 3. Данни от експеримент](#chast-3)
7. [Част 4. От записи към примери](#chast-4)
8. [Част 5. Модел и стратегия](#chast-5)
9. [Част 6. Изборът на етикет](#chast-6)
10. [Част 7. Наблюдение и експеримент](#chast-7)
11. [Обобщение](#obobshtenie)

<a id="cel"></a>
## Цел на упражнението

Упражнение 2 започваше от готови данни: таблици, изображения, времеви
редове, текст. Тази част започва една стъпка по-рано: **как се получават
данни**, когато готова таблица няма. Примерът е игра, дилемата на
затворника, а въпросът е:

**Какви данни са нужни, за да се научи модел да играе тази игра, и откъде
да се вземат?**

| Стъпка | Какво се прави | Къде |
|---|---|---|
| Играта | правилата, защо е дилема, какво променя повторението | част 1 |
| Стратегии | стратегията е функция, турнир като този на Axelrod | част 2 |
| Данни | какво е пример, какъв е етикетът, кой играе, какво се записва | част 3 |
| Примери | от записите на мачовете към таблица с решения и етикети | част 4 |
| Модел | средният етикет за всяка ситуация и ход, линейна регресия с качествен предиктор, стратегия от модела | част 5 |
| Етикет | колко рунда напред трябва да обхваща етикетът | част 6 |
| Наблюдение | защо записите от турнира подвеждат, а случайните ходове не | част 7 |

Моделът е линейната регресия от Тема 3 с качествен предиктор.
Данните са **симулирани**: те са
записи от турнири, изиграни в тетрадката. Така е работил и Robert Axelrod,
когато през 1980 г. е организирал първия компютърен турнир по тази игра.

<a id="failove"></a>
## Файлове в упражнението

| Файл | За какво служи |
|---|---|
| `Упражнение_3_част_1.ipynb` | обяснения, примери и задачи ДА ОПИТАМЕ |
| `ЗА_ВКЪЩИ_част_1.ipynb` | задачи за самостоятелна работа |
| `data/ipd_random_games.csv` | записите на 2400 мача от част 3; файлът се създава от тетрадката |
| `solution_1.py` | решения на задачите ДА ОПИТАМЕ |

<a id="podgotovka"></a>
## Подготовка и стартиране

Използваме общата среда от предишните упражнения и ядрото "Python (МС 26/27)".
Тетрадката се изпълнява самостоятелно от началото. Нужни са NumPy, Pandas,
Matplotlib и scikit-learn, които вече са включени в средата.

Стартирайте JupyterLab от папката на упражнение 3, за да работят
относителните пътища към папката `data/`. Клетките в [точка 6 "Партия срещу скрита стратегия"](#t6) и [точка 26 "Партия срещу модела"](#t26) чакат
въвеждане от клавиатурата.

In [ ]:
import solution_1

%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from matplotlib.colors import LinearSegmentedColormap
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder

DATA_DIR = Path("data")
BLUE, ORANGE = "#2a78d6", "#eb6834"
SEQUENTIAL = LinearSegmentedColormap.from_list("sequential", ["#f4f7fc", BLUE])
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
rng = np.random.default_rng(42) # генераторът на случайни числа за всички мачове
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

<a id="chast-1"></a>
## Част 1. Дилемата на затворника

<a id="t1"></a>
### 1. Историята на играта

**Откъде идва играта.** След Втората световна война RAND Corporation в
Калифорния е изследователски център, който работи за военновъздушните
сили на САЩ. Там математици изследват теорията на игрите: как се вземат
решения, когато резултатът зависи и от решенията на другите. В началото
на 1950 г. Merrill Flood и Melvin Dresher от RAND измислят проста игра за
двама. Всеки избира едно от две действия, без да знае избора на другия, а
печалбата му зависи и от двата избора. Играта е построена така, че всеки
поотделно печели повече, ако не сътрудничи с другия, а двамата заедно
печелят повече, ако сътрудничат помежду си. Тук и навсякъде по-нататък
"сътрудничество" означава сътрудничество **между двамата играчи**.

Flood и Dresher карат двама свои колеги да изиграят тази игра 100 пъти
подред. Според теорията разумните играчи не трябва да сътрудничат. Двамата
колеги обаче сътрудничат помежду си в много от рундовете. Причината е повторението:
след всеки рунд всеки вижда какво е избрал другият и може да го накаже
или да го награди в следващия рунд. John Nash, чиято теория е била
проверявана, възразява, че 100 поредни рунда не са 100 отделни игри, а
една голяма игра с други правила. Именно тази повтаряща се игра се
изследва в тетрадката.

**Историята със затворниците.** По-късно през 1950 г. математикът Albert
Tucker от Princeton (научният ръководител на Nash) трябва да изнесе лекция
пред психолози в Stanford. За да обясни играта на хора без математическа
подготовка, той я облича в история за двама затворници. От тази история
идва и името "дилема на затворника", което обикновено се приписва на
Tucker. Историята е следната.

Двама заподозрени са арестувани за общо престъпление и са разпитвани в
отделни стаи, без да могат да се договорят. Доказателствата стигат само за
малко наказание, затова на всеки се предлага сделка: ако свидетелства
срещу другия, ще бъде пуснат.

- Ако двамата мълчат, всеки получава 1 година за по-малкото обвинение.
- Ако единият свидетелства, а другият мълчи, свидетелят е свободен, а
  мълчалият получава 10 години.
- Ако двамата свидетелстват, всеки получава 5 години.

**Един ход.** При една среща всеки от двамата прави един **ход**: избира
една от две букви.

- **C** (cooperate, сътрудничество): затворникът мълчи, т.е. сътрудничи
  с другия затворник, като не го издава;
- **D** (defect, предателство): затворникът свидетелства срещу другия,
  т.е. го предава.

C не е сътрудничество с полицията. Който сътрудничи с полицията,
свидетелства срещу другия, а това е D. В играта сътрудничеството е винаги
между двамата играчи: C означава "не издава другия", а D означава
"издава другия".

Двамата избират едновременно и тайно: никой не знае какво е избрал
другият, когато прави своя избор. Едва след като и двамата са избрали,
двата хода се разкриват и всеки получава точки.

**От години към точки.** В историята по-малко години е по-добре. В играта
се използват **точки**, при които повече е по-добре. Четирите изхода
запазват реда си от най-добрия към най-лошия за играча. Във всеки изход
първо е играчът, а после другият:

| Ходът на играча и на другия | В историята | Точки на играча и на другия | Името на точките на играча |
|---|---|---|---|
| D и C | свободен и 10 години | 5 и 0 | T (temptation): изкушението на предателството |
| C и C | 1 година и 1 година | 3 и 3 | R (reward): награда за взаимното сътрудничество |
| D и D | 5 години и 5 години | 1 и 1 | P (punishment): наказание за взаимното предателство |
| C и D | 10 години и свободен | 0 и 5 | S (sucker): точките на излъгания |

Редовете "D и C" и "C и D" са един и същ изход, видян от двете страни:
когато единият получава T, другият получава S.

Самите числа не се изчисляват от годините. Важно е само подреждането им и
едно допълнително условие, което се вижда в [точка 2 "Защо е дилема"](#t2). Числата 5, 3, 1 и 0 са
тези от турнирите на Axelrod и оттогава са стандартни.

**Как играта стига до компютрите.** През 1950-те и 1960-те години
играта се изследва с хора в лаборатории: двама души играят много рундове
един срещу друг, а изследователите записват избора им. Една от
най-известните книги за такива опити е "Prisoner's Dilemma" (1965) на
психолога и математик Anatol Rapoport и Albert Chammah. Опитите с хора
обаче показват как играят хората, а не коя стратегия е най-добра.

Robert Axelrod, политолог от University of Michigan, задава точно този
въпрос и през 1980 г. сменя хората с компютърни програми. За да може
играта да се играе от компютър, тя се променя така:

| В историята и в опитите с хора | В компютърния турнир |
|---|---|
| години в затвора, по-малко е по-добре | точки, повече е по-добре (таблицата горе) |
| в историята затворниците се срещат веднъж | мач от определен брой рундове; след всеки рунд ходовете се разкриват ([точка 3 "Повтаряща се игра"](#t3)) |
| човекът решава в момента и може да промени начина си на игра | стратегията е програма, написана предварително: получава ходовете от изиграните рундове и връща C или D ([точка 4 "Стратегия"](#t4)) |
| двама души | турнир: всяка програма играе срещу всяка друга, а точките от всички мачове се събират ([точка 7 "Турнирите на Axelrod"](#t7)) |

Програмата не преговаря, не се ядосва и не се уморява. Тя прилага едно и
също правило всеки път. Затова мачът може да се изиграе отново със
същия резултат (при случайните стратегии със същото семе), а различните
стратегии могат да се сравнят точно. Така турнирът става източник на
**данни**, а това е темата на част 3.

В кода точките са речник. Ключът е ходът на играча, последван от хода на
другия: `"DC"` означава, че играчът е предал, а другият е сътрудничил.

In [ ]:
POINTS = {"CC": 3, "CD": 0, "DC": 5, "DD": 1} # ходът на играча и ходът на другия

table = pd.DataFrame( # във всяка клетка: (точки на играча, точки на другия)
    [[(POINTS["CC"], POINTS["CC"]), (POINTS["CD"], POINTS["DC"])],
     [(POINTS["DC"], POINTS["CD"]), (POINTS["DD"], POINTS["DD"])]],
    index=["играчът играе C", "играчът играе D"],
    columns=["другият играе C", "другият играе D"])
table

Играчът избира реда, а другият избира колоната. Във всяка клетка първото
число е точките на играча, а второто е точките на другия. Двамата са в
едно и също положение, затова един речник стига и за двамата: точките на
другия се взимат от `POINTS` с разменени ходове. Например, когато играчът
играе C, а другият D, играчът получава `POINTS["CD"]`, т.е. 0 точки, а
другият получава `POINTS["DC"]`, т.е. 5 точки.

<a id="t2"></a>
### 2. Защо е дилема

Ето какво вижда всеки от двамата, ако мисли само за този ход:

In [ ]:
for other in "CD":
    print(f"Другият играе {other}: C носи {POINTS['C' + other]} точки, D носи {POINTS['D' + other]} точки")
print()
print("Двамата играят C:", POINTS["CC"], "+", POINTS["CC"], "=", 2 * POINTS["CC"], "точки общо")
print("Двамата играят D:", POINTS["DD"], "+", POINTS["DD"], "=", 2 * POINTS["DD"], "точки общо")

Каквото и да играе другият, D носи с 2 точки повече. Затова играч, който
мисли само за себе си, играе D. Другият разсъждава по същия начин и също
играе D. Двамата получават по 1 точка, а можеха да получат по 3.
**Разумното решение за всеки поотделно е лошо за двамата заедно.** Това е
дилемата.

Една игра е дилема на затворника, ако точките спазват две условия:

- T > R > P > S: изкушението е по-голямо от наградата, а взаимното
  предателство е по-добро от това играчът да бъде излъган;
- 2R > T + S: редуването на C и D не е по-изгодно от постоянното
  сътрудничество. Ако двамата се редуват да се лъжат (CD, DC, CD, ...),
  всеки взема средно (5 + 0) / 2 = 2.5 точки на ход, а при сътрудничество
  взема 3.

Всякакви точки, които спазват двете условия, дават същата игра, например
3, 2, 1 и 0.

In [ ]:
T, R, P, S = POINTS["DC"], POINTS["CC"], POINTS["DD"], POINTS["CD"]
print("T > R > P > S:", T > R > P > S)
print("2R > T + S:", 2 * R > T + S)

Същата структура има в много ситуации:

- две бензиностанции на един път: ако едната намали цената, взема
  клиентите на другата, а ако и двете намалят, губят и двете;
- надпревара във въоръжаването;
- допинг в спорта;
- общ проект в екип: всеки предпочита другите да свършат работата;
- окопите от Първата световна война, където противникови части на места
  са спазвали неписаното правило "живей и остави другите да живеят".

<a id="t3"></a>
### 3. Повтаряща се игра

При една среща D винаги е по-изгодно ([точка 2 "Защо е дилема"](#t2)). Играта, която се изследва
тук, е друга: двамата се срещат много пъти подред. Една среща се нарича
**рунд**, а поредицата от рундове между двама играчи е **мач**. Правилата
на мача са:

1. Мачът има определен брой рундове. Тук те са 100.
2. Във всеки рунд всеки играч прави по един ход: C или D. Двамата избират
   едновременно, без да знаят какво избира другият в този рунд.
3. След рунда двата хода се разкриват. Всеки научава какво е избрал
   другият и получава точките си по таблицата от [точка 1 "Историята на играта"](#t1).
4. В следващия рунд всеки може да избере хода си според всичко, случило
   се дотук: своите ходове и ходовете на другия във всички изиграни
   рундове.
5. Точките от всички рундове се събират. Всеки играч се стреми да събере
   колкото може повече точки за себе си. Целта не е да има повече точки
   от другия.

**Защо целта не е да се победи другият.** В повечето игри, например в
шаха, единият печели, а другият губи. Тук не е така. Точките не се вземат
от другия, а се раздават по таблицата, както банка изплаща пари и на
двамата. Когато двамата сътрудничат, всеки получава по 3 точки и никой не
губи. Затова резултатът на играча е колко точки е събрал, а не дали е
събрал повече от другия. Пример с мач от 100 рунда:

- двамата играят C във всички рундове: всеки има по 300 точки и никой не
  е "победил";
- единият играе D във всички рундове, а другият започва с C и после
  отговаря с D до края: предателят има повече точки от другия, 104 срещу
  99, но е събрал почти три пъти по-малко, отколкото при сътрудничество.

Така е и в турнира на Axelrod ([точка 7 "Турнирите на Axelrod"](#t7)):
класирането е по сбора от точките от всички мачове, а не по броя
спечелени мачове. Затова една стратегия може да спечели турнира, без да
спечели нито един мач ([точка 10 "Победител без победа"](#t10)).

Всеки играч прави точно един ход във всеки рунд, затова "точки на ход" и
"точки на рунд" по-нататък означават едно и също.

Пример с 4 рунда: играч A започва с C и после повтаря последния ход на B,
а B избира както иска.

| Рунд | Какво знаят двамата преди рунда | Ход на A | Ход на B | Точки на A | Точки на B |
|---|---|---|---|---|---|
| 1 | нищо, мачът тепърва започва | C | D | 0 | 5 |
| 2 | рунд 1: A е играл C, B е играл D | D | C | 5 | 0 |
| 3 | рундове 1 и 2: A е играл C, D, а B е играл D, C | C | C | 3 | 3 |
| 4 | рундове 1 до 3: A е играл C, D, C, а B е играл D, C, C | C | C | 3 | 3 |
| общо | | | | 11 | 11 |

В рунд 2 играч A отвръща на предателството на B от рунд 1, а в рунд 3 се
връща към C, защото B е сътрудничил в рунд 2. Това е възможно само защото
след всеки рунд ходовете се разкриват. Така днешното предателство може да
струва утрешното сътрудничество. Axelrod нарича това "сянката на
бъдещето".

In [ ]:
ROUNDS = 100 # рундове в един мач

print("C от двамата във всичките 100 рунда:", ROUNDS * POINTS["CC"], "точки за всеки")
print("Предателство в първия рунд, после наказание до края:",
      POINTS["DC"] + (ROUNDS - 1) * POINTS["DD"], "точки за предателя")

Едно предателство носи 5 точки вместо 3. Но ако другият отговаря с D във
всички следващи рундове, предателят получава общо 104 точки вместо 300. В
повтарящата се игра има смисъл да се сътрудничи и да се наказва.
Въпросът вече не е кой ход е по-добър в един рунд, а **кое правило за
избор на ходове** е по-добро в целия мач.

<a id="t4"></a>
### 4. Стратегия

Стратегията е правило, което избира хода в следващия рунд според
изиграните дотук рундове. В кода тя е функция с два аргумента: списък с
ходовете на играча в изиграните рундове и списък с ходовете на другия в
същите рундове. В първия рунд двата списъка са празни. Функцията връща
`"C"` или `"D"`. Ходът на другия в текущия рунд не е сред аргументите,
защото двамата избират едновременно. Ето четири прости стратегии:

In [ ]:
def always_c(my, their):
    """Винаги сътрудничи."""
    return "C"


def always_d(my, their):
    """Винаги предава."""
    return "D"


def random_move(my, their):
    """Хвърля монета."""
    return "C" if rng.random() < 0.5 else "D"


def tit_for_tat(my, their):
    """Започва с C, после повтаря последния ход на другия."""
    return their[-1] if their else "C"


print("Tit for Tat в първия рунд:", tit_for_tat([], []))
print("Tit for Tat след два рунда, в които другият е играл C и D:", tit_for_tat(["C", "C"], ["C", "D"]))

Tit for Tat ("око за око") е най-известната стратегия. Тя никога не
предава първа, наказва веднага, но и прощава веднага, щом другият се
върне към C.

<a id="t5"></a>
### 5. Мач

Функцията `play` изиграва мач рунд по рунд. Във всеки рунд двете
стратегии получават само ходовете от изиграните рундове и избират. Чак
след като и двете са избрали, двата хода се добавят към историята, т.е.
се разкриват. Така никоя стратегия не вижда хода на другата в текущия
рунд.

`play` връща ходовете на двете като низове: i-тата буква е ходът в i-тия
рунд, например `"CCDC..."`.

In [ ]:
def play(strategy_a, strategy_b, rounds=ROUNDS):
    """Мач между две стратегии. Връща ходовете на двете като низове."""
    moves_a, moves_b = [], []
    for _ in range(rounds):                # един рунд
        a = strategy_a(moves_a, moves_b)   # всяка стратегия получава първо своите ходове
        b = strategy_b(moves_b, moves_a)   # b не знае хода a от този рунд
        moves_a.append(a)                  # двата хода се разкриват
        moves_b.append(b)
    return "".join(moves_a), "".join(moves_b)


def points(moves, opp_moves):
    """Точките на първия играч в мача."""
    return sum(POINTS[a + b] for a, b in zip(moves, opp_moves))


a, b = play(tit_for_tat, always_d, rounds=8)
display(pd.DataFrame({"Tit for Tat": list(a), "Always D": list(b)},
                     index=pd.Index(range(1, 9), name="рунд")).T)
print("Точки за 8 рунда:", points(a, b), "и", points(b, a))

Tit for Tat губи само първия рунд: 0 срещу 5 точки. От втория рунд
нататък двамата играят D и вземат по 1 точка. В мач от 100 рунда това са
99 срещу 104 точки. Tit for Tat никога не печели мач: в най-добрия случай завършва
наравно. Това ще има значение в част 2.

<a id="t6"></a>
### 6. Партия срещу скрита стратегия

Най-лесно играта се разбира, като се изиграе. Клетката избира скрито една
от четирите стратегии и започва мач от 10 рунда. Във всеки рунд въведете C
или D (малка или голяма буква). Стратегията избира своя ход, без да
знае хода на човека в този рунд, точно както в `play`. След всеки рунд се
показват двата хода и точките. След последния рунд стратегията се
разкрива. Коя е била и как личи от ходовете ѝ?

In [ ]:
def play_against(strategy, rounds=10, name=None):
    """Човек срещу стратегия. Човекът въвежда C или D."""
    human, machine = [], []
    for k in range(1, rounds + 1):
        move = strategy(machine, human) # стратеdгията избира, без да знае хода на човека в този рунд
        answer = input(f"Рунд {k}, C или D: ").strip().upper()
        while answer not in ("C", "D"):
            answer = input("Само C или D: ").strip().upper()
        human.append(answer)
        machine.append(move)
        print(f"   човек {answer}, машина {move}: {POINTS[answer + move]} и {POINTS[move + answer]} точки")
    print("Общо: човек", points(human, machine), "точки, стратегия", points(machine, human), "точки")
    if name:
        print("Стратегията беше:", name)


hidden = {"Always C": always_c, "Always D": always_d, "Random": random_move, "Tit for Tat": tit_for_tat}
secret = str(rng.choice(list(hidden)))
play_against(hidden[secret], name=secret)

<a id="chast-2"></a>
## Част 2. Стратегии и турнир

<a id="t7"></a>
### 7. Турнирите на Axelrod

За първия си турнир (резултатите са публикувани през 1980 г.) Robert
Axelrod кани специалисти по теория на игрите, икономисти, психолози и
математици да изпратят стратегия като компютърна програма. Получава 14
програми и добавя към тях Random. Всяка програма играе срещу всяка друга
и срещу себе си мач от 200 рунда, а турнирът е повторен 5 пъти. Печели
най-кратката програма: Tit for Tat на Anatol Rapoport, който дотогава
години наред е изследвал играта с хора.

Axelrod публикува резултатите и организира втори турнир. Този път
участниците са 62 от шест държави и всички знаят кой е спечелил първия.
Rapoport изпраща отново Tit for Tat и печели отново.

Изводите на Axelrod за успешните стратегии са четири: да са **добри**
(никога не предават първи), да **отвръщат** на предателство, да
**прощават**, когато другият се върне към C, и да са **ясни**, за да може
другият да разбере правилото им.

<a id="t8"></a>
### 8. Известните стратегии

Тук турнирът е с 12 стратегии. Някои са участвали при Axelrod, други са
по-късни.

| Стратегия | Правило |
|---|---|
| Always C | винаги C |
| Always D | винаги D |
| Random | C или D с вероятност 1/2 |
| Tit for Tat | започва с C, после повтаря последния ход на другия |
| Suspicious TFT | като Tit for Tat, но започва с D |
| Tit for Two Tats | D само след два поредни D на другия; според Axelrod би спечелила първия турнир, но във втория се класира слабо |
| Grudger | C, докато другият не предаде веднъж, после D до края; в първия турнир е програмата на James Friedman |
| Pavlov | ако другият е играл C, повтаря своя последен ход, иначе го сменя ("win-stay, lose-shift") |
| Joss | като Tit for Tat, но в 10% от случаите отговаря на C с D; участник в първия турнир |
| Alternator | редува C и D |
| Hard TFT | D, ако другият е предал в някой от последните три рунда |
| Detective | започва с C, D, C, C; ако другият не е отвърнал с D, предава до края, иначе играе Tit for Tat |

In [ ]:
def suspicious_tft(my, their):
    return their[-1] if their else "D"


def tit_for_two_tats(my, their):
    return "D" if their[-2:] == ["D", "D"] else "C"


def grudger(my, their):
    return "D" if "D" in their else "C"


def pavlov(my, their):
    if not my:
        return "C"
    if their[-1] == "C":
        return my[-1]
    return "D" if my[-1] == "C" else "C"


def joss(my, their):
    move = tit_for_tat(my, their)
    return "D" if move == "C" and rng.random() < 0.1 else move


def alternator(my, their):
    return "C" if len(my) % 2 == 0 else "D"


def hard_tft(my, their):
    return "D" if "D" in their[-3:] else "C"


def detective(my, their):
    if len(my) < 4:
        return "CDCC"[len(my)]
    return their[-1] if "D" in their[:4] else "D"


STRATEGIES = {"Always C": always_c, "Always D": always_d, "Random": random_move,
              "Tit for Tat": tit_for_tat, "Suspicious TFT": suspicious_tft,
              "Tit for Two Tats": tit_for_two_tats, "Grudger": grudger, "Pavlov": pavlov,
              "Joss": joss, "Alternator": alternator, "Hard TFT": hard_tft, "Detective": detective}
print("Известни стратегии:", len(STRATEGIES))

<a id="t9"></a>
### 9. Турнир

Функцията `tournament` пуска всеки играч срещу всеки противник по няколко
мача и връща **таблица със записи**: по един ред на мач, с ходовете на
двамата. Точките не се записват, защото се изчисляват от ходовете.
Турнирът на известните стратегии е като този на Axelrod: всяка срещу
всяка, включително срещу себе си, по 5 мача.

In [ ]:
def tournament(players, opponents, matches=1):
    """Всеки играч срещу всеки противник. Един ред на мач."""
    rows = []
    for name, strategy in players.items():
        for opp_name, opponent in opponents.items():
            for match in range(1, matches + 1):
                moves, opp_moves = play(strategy, opponent)
                rows.append({"player": name, "opponent": opp_name, "match": match,
                             "moves": moves, "opp_moves": opp_moves})
    return pd.DataFrame(rows)


rng = np.random.default_rng(1980)
games = tournament(STRATEGIES, STRATEGIES, matches=5)
games["points"] = [points(a, b) for a, b in zip(games["moves"], games["opp_moves"])]
games["opp_points"] = [points(b, a) for a, b in zip(games["moves"], games["opp_moves"])]
print("Мачове:", len(games))
games.sample(5, random_state=0) # 5 произволни мача

In [ ]:
games["score"] = games["points"] / ROUNDS            # средни точки на ход
games["won"] = games["points"] > games["opp_points"] # спечелен мач

ranking = (games.groupby("player")
                      .agg(points_per_move=("score", "mean"),
                           matches_won=("won", "sum"),
                           total_points=("points", "sum"))   # общо точки от всички мачове
                      .sort_values("points_per_move", ascending=False))
ranking.round(3)

In [ ]:
order = ranking.index
grid = games.pivot_table(index="player", columns="opponent", values="score").loc[order, order]

fig, ax = plt.subplots(figsize=(8, 6.5))
image = ax.imshow(grid, cmap=SEQUENTIAL, vmin=0, vmax=5)
ax.set_xticks(range(len(order)), order, rotation=60, ha="right")
ax.set_yticks(range(len(order)), order)
for i in range(len(order)):
    for j in range(len(order)):
        value = grid.iloc[i, j]
        ax.text(j, i, f"{value:.1f}", ha="center", va="center", fontsize=8,
                color="white" if value > 3.2 else "#333333")
ax.set_xlabel("противник")
ax.set_title("Средни точки на ход за стратегията в реда")
fig.colorbar(image, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

<a id="t10"></a>
### 10. Победител без победа

Tit for Tat е първа, без да е спечелила нито един от своите 60 мача.
Always D печели най-много мачове (49 от 60), но е предпоследна. В турнира точките
не се вземат от другия, а **заедно с другия**: Tit for Tat взема по 3
точки на ход от всяка стратегия, която сътрудничи, и губи малко срещу
тези, които предават. Always D печели всяка среща, но срещу почти всички
получава по 1 точка на ход.

In [ ]:
# --- ДА ОПИТАМЕ 1 ---
#
# Напишете своя стратегия като функция my, their -> "C" или "D" и я добавете
# към турнира: players = {**STRATEGIES, "Нова": new_strategy}, после
# tournament(players, players, matches=5). На кое място се класира? Може ли
# да се класира по-високо от Tit for Tat?

solution_1.to_test_1(STRATEGIES, tournament, points)

<a id="chast-3"></a>
## Част 3. Данни от експеримент

Целта е модел, който се научава да играе: във всеки рунд да избира C или
D. За тази игра няма готова таблица с данни. Има само правилата и
възможността да се играят мачове. Данните трябва да се **създадат**, а
преди това трябва да се реши какви да бъдат: какво е пример, какъв е
етикетът, кой играе и какво се записва.

**Идеята накратко.** Моделът научава какво да се играе във всяка
ситуация. За целта гледа хиляди случаи, в които в тази ситуация е
изиграно C или D, и сравнява колко точки са събрани след това.

1. Когато избира хода си, играчът знае само какво са играли двамата в
   предишния рунд. Възможностите са четири: CC, CD, DC и DD. Към тях се
   добавя "начало" за първия рунд.
2. Затова стратегията може да е таблица с 5 реда: за всяка ситуация C
   или D. Tit for Tat например е C, C, D, C, D. Задачата на модела е да
   попълни тази таблица.
3. Кой ход е по-добър, се разбира с опити. Във всяка ситуация са нужни
   много случаи с C и много случаи с D. Опитите прави монета, т.е.
   Random, която играе наслуки срещу 12-те известни стратегии.
4. Резултатът от един ход е сборът от точките в този и в следващите 4
   рунда, защото наказанието за D идва след него.
5. За всяка ситуация се изчислява средният резултат при C и при D и се
   избира по-добрият ход. Така таблицата от стъпка 2 се попълва и
   стратегията е готова.
6. Стратегията се проверява в нови мачове срещу известните стратегии.

Частите 6 и 7 показват двете условия за данните: резултатът трябва да
обхваща следващите рундове, а ходовете трябва да са случайни.

<a id="t11"></a>
### 11. Пример за всяко решение

Във всеки рунд играчът взема едно решение: C или D. Затова **един пример
е едно решение**. Когато решава, играчът знае какво са играли двамата в
предишния рунд. Това е **ситуацията**. Тя се записва с две букви: първо
ходът на играча, после ходът на другия, както ключовете на `POINTS`.

| Ситуация | Какво е станало в предишния рунд |
|---|---|
| CC | двамата са сътрудничили |
| CD | играчът е сътрудничил, а другият е предал |
| DC | играчът е предал, а другият е сътрудничил |
| DD | двамата са предали |
| начало | това е първият рунд и предишен рунд няма |

Признаците на един пример са ситуацията и ходът на играча. **Етикетът**
показва колко добро е било решението. Първата мисъл е етикетът да са
точките от самия рунд. Но в един рунд D винаги носи с 2 точки повече от C
([точка 2 "Защо е дилема"](#t2)), а отговорът на другия идва едва в
следващия рунд ([точка 3 "Повтаряща се игра"](#t3)). Затова етикетът е
**сборът от точките на играча в този рунд и в следващите 4**, общо 5
рунда. Така в етикета влиза и реакцията на другия. Колко рунда са нужни,
се проверява в част 6.

Например играчът е в ситуация CD: в предишния рунд е бил излъган. Сега
играе D и в този и в следващите 4 рунда получава 5, 1, 1, 3, 3 точки.
Това е един пример: ситуация CD, ход D, етикет 13.

Моделът е прост. За всяка ситуация се изчислява средният етикет, когато
играчът е играл C, и средният етикет, когато е играл D. Във всяка
ситуация се избира ходът с по-високото средно. Така моделът казва какво
да се играе във всяка от петте ситуации, а това вече е **стратегия**.

<a id="t12"></a>
### 12. Кой играе

За да се сравнят C и D във всяка ситуация, са нужни много примери и с
двата хода. Кой да играе мачовете?

Първата възможност са записите от турнира в част 2. Там обаче всяка
стратегия играе по своето правило. Tit for Tat например почти никога не
играе D след CC, а Always D никога не играе C. Кой ход се среща в една
ситуация, зависи от това коя стратегия играе и срещу кого. Част 7 показва
какво научава моделът от такива данни.

Втората възможност е играч, който **хвърля монета**: Random от
[точка 8 "Известните стратегии"](#t8). Той играе C или D с вероятност
1/2 във всяка ситуация и срещу всеки противник. Затова във всяка ситуация
има приблизително еднакво много примери с C и с D, а срещу всеки
противник двата хода се срещат еднакво често. Така се провеждат и
изпитванията на лекарства: кой получава лекарството, решава случайността.

<a id="t13"></a>
### 13. Планът на експеримента

| Решение | Избор | Защо |
|---|---|---|
| пример | едно решение: един рунд на играча | моделът трябва да избира хода във всеки рунд |
| признаци | ситуацията и ходът | това знае и това избира играчът |
| етикет | точките на играча в този и в следващите 4 рунда | в етикета влиза и реакцията на другия |
| играч | Random, т.е. монета | двата хода се срещат във всяка ситуация срещу всеки противник |
| противници | 12-те известни стратегии | срещу тях моделът се учи да играе |
| мачове | по 200 срещу всеки противник, по 100 рунда | разликите между C и D са малки, затова са нужни много примери |
| записи | всички ходове на двамата | от тях се изчисляват и ситуациите, и етикетите |

Етикетът не се записва, а се изчислява от ходовете, както точките в
част 2.

<a id="t14"></a>
### 14. Записите

Турнирът се играе, а записите се пазят в CSV файл. Така анализът може да
се повтори, без турнирът да се играе отново, а файлът може да се даде на
друг. В истински турнир организаторът дава точно такъв файл.

In [ ]:
rng = np.random.default_rng(2027)
random_games = tournament({"Random": random_move}, STRATEGIES, matches=200)
random_games.to_csv(DATA_DIR / "ipd_random_games.csv", index=False)

print("Мачове:", len(random_games), " рундове на Random:", len(random_games) * ROUNDS)
print()
with open(DATA_DIR / "ipd_random_games.csv", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().rstrip()[:80] + "...")

<a id="chast-4"></a>
## Част 4. От записи към примери

<a id="t15"></a>
### 15. Проверка на записите

Записите се проверяват, преди да се използват, дори когато са създадени
от собствен код. Грешка в симулацията често се вижда по-лесно в данните,
отколкото в кода. Проверките тук са пет: брой мачове срещу всеки
противник, дължина на ходовете, само букви C и D, повторени редове и
дали монетата е честна, т.е. дали Random играе C в около половината
рундове.

In [ ]:
records = pd.read_csv(DATA_DIR / "ipd_random_games.csv")
print("Редове и колони:", records.shape)
print("Мачове срещу всеки противник:", records.groupby("opponent").size().unique())
print("Дължини на ходовете:", records["moves"].str.len().unique(), records["opp_moves"].str.len().unique())
print("Само C и D:", records["moves"].str.fullmatch("[CD]+").all() and records["opp_moves"].str.fullmatch("[CD]+").all())
print("Повторени редове:", records.duplicated().sum())
print("Дял на C в ходовете на Random:", round(records["moves"].str.count("C").sum() / (len(records) * ROUNDS), 3))

<a id="t16"></a>
### 16. От мач към рундове

Един ред е цял мач. За примерите е нужен всеки рунд поотделно, заедно с
двата хода от предишния рунд. Ето стъпките върху един кратък мач от 6
рунда:

In [ ]:
example = pd.DataFrame({"player": ["A"], "opponent": ["B"], "match": [1],
                        "moves": ["CCDCDD"], "opp_moves": ["CDCCDC"]})
rounds = example.assign(move=example["moves"].map(list), opp_move=example["opp_moves"].map(list))
rounds = rounds.explode(["move", "opp_move"])                    # един ред на рунд
rounds["round"] = rounds.groupby(level=0).cumcount() + 1          # номер на рунда в мача
rounds["my_prev"] = rounds.groupby(level=0)["move"].shift()       # ходът на играча в предишния рунд
rounds["opp_prev"] = rounds.groupby(level=0)["opp_move"].shift()  # ходът на другия в предишния рунд
rounds["situation"] = (rounds["my_prev"] + rounds["opp_prev"]).fillna("начало")
rounds[["round", "my_prev", "opp_prev", "situation", "move", "opp_move"]]

- `map(list)` превръща низа в списък от букви.
- `explode` прави всеки елемент на списъка отделен ред. Индексът остава
  номерът на мача.
- `groupby(level=0)` групира по индекса, т.е. по мач. `cumcount` номерира
  рундовете, а `shift` взема хода от предишния рунд **в същия мач**. Без
  групирането първият рунд на един мач би получил последния рунд на
  предишния мач.
- Първият рунд няма предишен рунд. `shift` му дава `NaN`, сборът на
  `NaN` с буква също е `NaN`, а `fillna` го заменя с "начало".

В този кратък мач играчът A минава през всичките пет ситуации.

<a id="t17"></a>
### 17. Етикетът

Точките за всеки рунд се вземат от `POINTS` по двете букви. Етикетът е
сборът от точките в този рунд и в следващите. `shift(-1)` взема точките
от следващия рунд в същия мач, `shift(-2)` от по-следващия и т.н. В
малкия пример етикетът обхваща 3 рунда, за да се вижда добре. В данните
ще обхваща 5.

In [ ]:
rounds["points"] = (rounds["move"] + rounds["opp_move"]).map(POINTS)
by_match = rounds.groupby(level=0)["points"]
rounds["future"] = rounds["points"] + by_match.shift(-1) + by_match.shift(-2) # този и следващите 2 рунда
rounds[["round", "situation", "move", "opp_move", "points", "future"]]

Рунд 3 е един пример: ситуация CD (в рунд 2 играчът A е сътрудничил, а B
е предал), ход D, етикет 5 + 3 + 1 = 9. Последните два рунда нямат
достатъчно следващи рундове и етикетът им е `NaN`. Такива редове се
махат. В мач от 100 рунда с етикет от 5 рунда остават 96 примера.

<a id="t18"></a>
### 18. Всички примери

Същите стъпки за всички записи са събрани във функцията `decisions`. Тя
връща по един ред на решение. Аргументът `horizon` казва колко рунда
обхваща етикетът.

In [ ]:
def decisions(games, horizon=5):
    """Един ред на решение: ситуация, ход и етикет от horizon рунда."""
    rounds = games.assign(move=games["moves"].map(list), opp_move=games["opp_moves"].map(list))
    rounds = rounds.explode(["move", "opp_move"])
    by_match = rounds.groupby(level=0)
    rounds["round"] = by_match.cumcount() + 1
    rounds["situation"] = (by_match["move"].shift() + by_match["opp_move"].shift()).fillna("начало")
    rounds["points"] = (rounds["move"] + rounds["opp_move"]).map(POINTS)
    by_match = rounds.groupby(level=0)["points"]
    rounds["future"] = sum(by_match.shift(-k) for k in range(horizon)) # този и следващите horizon - 1 рунда
    columns = ["player", "opponent", "match", "round", "situation", "move", "future"]
    return rounds.dropna(subset=["future"])[columns].reset_index(drop=True)


data = decisions(records)
print("Примери:", len(data), " на мач:", len(data) // len(records))
display(data["situation"].value_counts())
data.sample(5, random_state=42)

От всеки мач остават 96 примера, общо 230400. Ситуациите CD и DD са
по-чести от CC и DC, защото няколко противници често предават: Always D
винаги, а Grudger до края след първото D на монетата.

In [ ]:
# --- ДА ОПИТАМЕ 2 ---
#
# Срещу кои противници Random попада във всяка ситуация? Направете таблица с
# дела на всяка ситуация срещу всеки противник: pd.crosstab с
# normalize="index". Има ли ситуации, които идват само от някои противници?

solution_1.to_test_2(data)

<a id="chast-5"></a>
## Част 5. Модел и стратегия

<a id="t19"></a>
### 19. Средните за всяка ситуация

Моделът сравнява двата хода във всяка ситуация: средния етикет, когато
Random е играл C, и средния етикет, когато е играл D. `pivot_table`
изчислява всички средни наведнъж: редовете са ситуациите, а колоните са
ходовете.

In [ ]:
SITUATIONS = ["начало", "CC", "CD", "DC", "DD"]

averages = data.pivot_table(index="situation", columns="move", values="future", aggfunc="mean").loc[SITUATIONS]
averages["C - D"] = averages["C"] - averages["D"]
display(averages.round(2))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
colors = [BLUE if value > 0 else ORANGE for value in averages["C - D"]]
bars = ax.bar(averages.index, averages["C - D"], color=colors, width=0.6)
ax.bar_label(bars, fmt="%+.2f", padding=3)
ax.axhline(0, color="#888888", linewidth=1)
ax.set_xlabel("ситуация: двата хода в предишния рунд")
ax.set_ylabel("C - D: разлика в етикета")
ax.set_title("Колко точки повече носи C (синьо) или D (оранжево)")
ax.margins(y=0.2)
plt.tight_layout()
plt.show()

В четири от петте ситуации C носи повече. Само в CD, когато играчът
току-що е бил излъган, D носи повече: ако играчът не отвърне, някои
противници продължават да предават без наказание.

<a id="t20"></a>
### 20. Качествен предиктор в линейна регресия

Същите средни дава линейната регресия, обучена със scikit-learn с `fit`
и `predict`, както досега. В лекцията за линейната регресия има
**качествени предиктори**:

- **предиктор** е признакът, по който се предсказва, т.е. X;
- **качествен** значи, че стойностите са категории, а не числа, например
  пол или етническа група. Обратното е количествен предиктор, например
  доход или възраст.

Регресията работи с числа, затова категориите се кодират с
**индикаторни променливи** (dummy variable): 1, ако примерът е от
категорията, и 0, ако не е. Една категория остава **базова** и няма своя
променлива. При пола в данните Credit има една индикаторна променлива
(x = 1 за жена, x = 0 за мъж), а при три етнически групи има две. При q
категории са нужни q - 1 индикаторни променливи.

Тук предикторът е **случаят**: ситуацията и ходът заедно, например "CC C"
или "DD D". Случаите са 10 (5 ситуации по 2 възможности за ход, C или D), затова индикаторните
променливи са 9. Създава ги `OneHotEncoder`, същият като в Упражнение 2,
част 2. С `drop="first"` първата категория няма колона, т.е. тя е
базовата.

In [ ]:
data["case"] = data["situation"] + " " + data["move"] # ситуация и ход заедно: 10 категории
encoder = OneHotEncoder(drop="first", sparse_output=False).fit(data[["case"]])
names = encoder.get_feature_names_out()
print("Категории:", encoder.categories_[0].tolist())
print("Базова категория:", encoder.categories_[0][0])

X = pd.DataFrame(encoder.transform(data[["case"]]), columns=names).astype(int)
y = data["future"]
print("X:", X.shape, " y:", y.shape)
display(pd.concat([data[["case"]], X], axis=1).sample(5, random_state=0))

Във всеки ред има най-много една единица: в колоната на неговия случай.
Базовият случай "CC C" няма колона, затова неговите редове са само нули.

Моделът се обучава както досега: `fit` с X и етикетите y. После
`predict` дава прогноза за всеки от 10-те случая. Прогнозите се
подреждат в таблица като тази от
[точка 19 "Средните за всяка ситуация"](#t19).

In [ ]:
model = LinearRegression().fit(X, y)

cases = pd.DataFrame([(s, m) for s in SITUATIONS for m in ["C", "D"]], columns=["situation", "move"])
cases["case"] = cases["situation"] + " " + cases["move"]
X_cases = pd.DataFrame(encoder.transform(cases[["case"]]), columns=names)
cases["prediction"] = model.predict(X_cases)

predictions = cases.pivot(index="situation", columns="move", values="prediction").loc[SITUATIONS]
display(predictions.round(2))
print("Прогнозите съвпадат със средните от точка 19:", np.allclose(predictions, averages[["C", "D"]]))

Прогнозите на модела са точно средните. Линейната регресия избира
прогнозите с най-малка RSS. Тук X са само индикаторни променливи, затова
моделът дава една и съща прогноза за всички примери от един случай. За
група етикети с една обща прогноза RSS е най-малка, когато прогнозата е
средното на групата. Например при етикети 6 и 8 прогнозата 7 дава
RSS = 1 + 1 = 2, а прогнозата 6 или 8 дава RSS = 0 + 4 = 4. Затова
`pivot_table` и линейната регресия дават едни и същи числа.

<a id="t21"></a>
### 21. От таблицата към стратегия

Във всяка ситуация се избира ходът с по-голямото средно, т.е. с
по-голямата прогноза на модела. `idxmax(axis=1)`
връща за всеки ред името на колоната с най-голямата стойност. Получава
се речник с ход за всяка ситуация. Той е правило за игра, а функцията
`from_rule` го превръща в стратегия като тези от част 2.

In [ ]:
rule = averages[["C", "D"]].idxmax(axis=1).to_dict()
print("Правилото на модела:", rule)


def from_rule(rule):
    """Стратегия, която във всяка ситуация играе хода от правилото."""
    def strategy(my, their):
        situation = my[-1] + their[-1] if my else "начало"
        return rule[situation]
    return strategy


learned = from_rule(rule)
print("Модел срещу Tit for Tat:", play(learned, tit_for_tat, rounds=10))
print("Модел срещу Always D:   ", play(learned, always_d, rounds=10))

| Ситуация | Ход | Смисъл |
|---|---|---|
| начало | C | започва със сътрудничество |
| CC | C | продължава сътрудничеството |
| CD | D | отвръща, когато е излъган |
| DC | C | връща се към C, щом другият сътрудничи |
| DD | C | опитва да прекъсне взаимното предателство |

Това е почти Tit for Tat. Разликата е само в ситуацията DD: Tit for Tat
повтаря D, а моделът опитва C. Затова срещу Always D моделът редува C и
D. Моделът е стигнал до правилото сам, само от хвърлянията на монетата и
точките след тях. Правилата на играта и стратегиите на противниците не
са му казани.

<a id="t22"></a>
### 22. Проверка в турнир

Стратегията на модела и Tit for Tat играят по 30 нови мача срещу всяка
известна стратегия. Тези мачове не са сред примерите, от които моделът е
учил.

In [ ]:
def tournament_score(strategy, matches=30):
    """Средни точки на ход срещу всяка от 12-те известни стратегии."""
    trial = tournament({"проба": strategy}, STRATEGIES, matches=matches)
    trial["score"] = [points(a, b) / ROUNDS for a, b in zip(trial["moves"], trial["opp_moves"])]
    return trial.groupby("opponent")["score"].mean()


rng = np.random.default_rng(7)
check = pd.DataFrame({"Модел": tournament_score(learned), "Tit for Tat": tournament_score(tit_for_tat)})
display(check.round(2))
print("Средно:", check.mean().round(3).to_dict())

Срещу повечето стратегии двете играят еднакво. Моделът печели много
срещу Joss: Joss понякога предава без повод, Tit for Tat отвръща и двете
влизат в ехо от взаимни наказания, а моделът го прекъсва с C след DD.
Губи срещу Always D и Random, които не отговарят на това C. Средно
моделът е малко пред Tit for Tat.

Най-удачната проверка е турнирът от част 2 с още един участник:
стратегията на модела. Сега всички играят и срещу нея.

In [ ]:
rng = np.random.default_rng(1980)
players = {**STRATEGIES, "Модел": learned}
final = tournament(players, players, matches=5)
final["score"] = [points(a, b) / ROUNDS for a, b in zip(final["moves"], final["opp_moves"])]
final.groupby("player")["score"].mean().sort_values(ascending=False).round(3)

Моделът е първи, малко пред Tit for Tat. Той не е по-умен от Tit for
Tat: научил е какво носи точки срещу тези 12 противници. Срещу други
противници правилото може да е друго.

In [ ]:
# --- ДА ОПИТАМЕ 3 ---
#
# Дали моделът е намерил най-доброто правило? Правилото избира по един ход за
# всяка от петте ситуации, затова правилата, които гледат само предишния рунд,
# са 2 ** 5 = 32. Проверете всички 32 с tournament_score, както в точка 22, и
# намерете мястото на правилото на модела. Всички комбинации от ходове дава
# itertools.product("CD", repeat=5).

solution_1.to_test_3(rule, from_rule, tournament_score, rng)

<a id="chast-6"></a>
## Част 6. Изборът на етикет

<a id="t23"></a>
### 23. Колко рунда напред

Етикетът обхваща 5 рунда. Какво научава моделът, ако етикетът обхваща
по-малко или повече рундове? Функцията `learn` събира стъпките от точки
19 и 21: средните и хода с по-голямото средно. За всяка дължина на
етикета се учи ново правило и то се проверява в турнир.

In [ ]:
def learn(data):
    """Правилото на модела: във всяка ситуация ходът с по-голям среден етикет."""
    averages = data.pivot_table(index="situation", columns="move", values="future", aggfunc="mean")
    return averages.idxmax(axis=1).to_dict()


rows = []
for horizon in [1, 2, 3, 5, 10, 20]:
    rule_h = learn(decisions(records, horizon))
    rng = np.random.default_rng(7)
    rows.append({"рундове в етикета": horizon, **{s: rule_h[s] for s in SITUATIONS},
                 "точки на ход в турнир": tournament_score(from_rule(rule_h)).mean()})
pd.DataFrame(rows).set_index("рундове в етикета").round(3)

С етикет от 1 рунд моделът научава "винаги D", точно както се очакваше в
[точка 11 "Пример за всяко решение"](#t11): в един рунд D винаги носи
повече. В турнира това е Always D, една от последните стратегии. С 2
рунда правилото е странно: D след CC. От 3 рунда нататък правилото е едно
и също: това от точка 21.

**Какво научава моделът, зависи от това как е определен етикетът.**
Етикетът трябва да обхваща последствията от решението. Моделът и данните
са едни и същи, а различният етикет дава различна стратегия.

<a id="chast-7"></a>
## Част 7. Наблюдение и експеримент

<a id="t24"></a>
### 24. Данни от турнира

Защо е нужна монетата? Турнирът от част 2 вече има записи: 720 мача на
12-те известни стратегии. Ето какво научава моделът от тях със същия
етикет от 5 рунда. Таблицата показва колко примера има за всяка ситуация
и ход.

In [ ]:
observed = decisions(games)
display(observed.pivot_table(index="situation", columns="move", values="future", aggfunc="size").loc[SITUATIONS])
rule_observed = learn(observed)
print("Правилото от турнира:", rule_observed)
rng = np.random.default_rng(7)
print(f"Точки на ход в турнир: {tournament_score(from_rule(rule_observed)).mean():.3f}")

Правилото от турнира играе D след CC, т.е. предава, когато двамата
сътрудничат, и събира много по-малко точки. Таблицата показва и колко
неравни са данните: след CC има над 25000 примера с C и под 2000 с D.

<a id="t25"></a>
### 25. Защо наблюдението подвежда

Кой играе D след CC в записите от турнира и срещу кого?

In [ ]:
cc_d = observed[(observed["situation"] == "CC") & (observed["move"] == "D")]
display(cc_d["player"].value_counts())
cc_d.groupby("opponent")["future"].agg(["size", "mean"]).sort_values("size", ascending=False).round(1)

Повечето D след CC са на Alternator, който редува C и D независимо от
ситуацията, и на Random. Много от тези D са срещу Always C и Tit for Two
Tats, които не наказват едно D, и там етикетът е около 20. Затова
средното за D след CC е високо. То обаче показва кой срещу кого е играл,
а не какво носи D след CC. Срещу Tit for Tat или Grudger същото D струва
скъпо, но в записите такива примери са малко: тези стратегии и
противниците им рядко предават след CC.

Монетата прекъсва тази връзка. Random играе D след CC еднакво често
срещу всеки противник, затова средните сравняват само хода. Това е
разликата между **наблюдение** и **експеримент**: при наблюдението кой
ход се играе зависи от това кой играе и срещу кого, а при експеримента
го решава случайността.

In [ ]:
# --- ДА ОПИТАМЕ 4 ---
#
# Разликите между C и D в точка 19 са части от точката. Колко мача са нужни,
# за да се научава надеждно едно и също правило? Повторете експеримента от
# част 3 по 10 пъти с 10, 50, 200 и 500 мача срещу всеки противник и
# пребройте колко пъти се получава всяко правило.

solution_1.to_test_4(random_move, tournament, decisions, STRATEGIES)

<a id="t26"></a>
### 26. Партия срещу модела

Стратегията на модела е готова за игра. Въведете C или D за всеки от
10-те рунда. Колко точки може да се вземат срещу нея и как?

In [ ]:
play_against(learned)

<a id="obobshtenie"></a>
## Обобщение

- **Играта:** в един рунд D винаги носи повече, но двамата заедно губят. В
  повтарящата се игра сътрудничеството и наказанието имат смисъл.
- **Стратегия и турнир:** стратегията е функция от изиграните рундове.
  Турнирът дава таблица със записи: един ред на мач, с ходовете на
  двамата. Tit for Tat печели турнира, без да спечели мач.
- **Пример и етикет:** един пример е едно решение: ситуацията от
  предишния рунд и ходът. Етикетът са точките в този и в следващите
  рундове, защото последствията от хода идват по-късно.
- **Данни от експеримент:** играчът хвърля монета, затова двата хода се
  срещат във всяка ситуация срещу всеки противник. Записват се всички
  ходове, а семето прави симулацията повторима.
- **От записи към примери:** проверка на записите, `explode`,
  `groupby(level=0)` с `cumcount` и `shift`, `fillna` за първия рунд,
  `map` за точките и `shift(-k)` за етикета.
- **Модел и стратегия:** средният етикет за всяка ситуация и ход е същото
  като прогнозата на линейна регресия с качествен предиктор: ситуацията и
  ходът заедно, кодирани с индикаторни променливи. Ходът с по-голямото средно във
  всяка ситуация е правило за игра. Правилото е почти Tit for Tat и в
  турнира е малко пред нея.
- **Изборът на етикет:** с етикет от един рунд моделът научава "винаги D".
  Какво научава моделът, зависи от това как е определен етикетът.
- **Наблюдение и експеримент:** от записите на турнира моделът научава
  слабо правило, защото там кой ход се играе зависи от това кой играе и
  срещу кого. Монетата прекъсва тази връзка.